# Dirty iris: walidacja i czyszczenie z Pandera

W tym osobnym notebooku przechodzimy przez dwa kroki: najpierw walidację brudnego zbioru `dirty_iris`, a potem jego czyszczenie i ponowną kontrolę poprawności.

In [2]:
%pip install pandera -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
import numpy as np
import pandas as pd
import pandera as pa
from pandera import Check, Column, DataFrameSchema


## 1. Wczytanie danych

Najpierw ładujemy surowy plik i sprawdzamy kilka pierwszych wierszy.

In [26]:
dirty_iris = pd.read_csv("data/dirty_iris.csv")
dirty_iris.head(5)

,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
0,6.4,3.2,4.5,1.5,versicolor
1,6.3,3.3,6.0,2.5,virginica
2,6.2,NaN,5.4,2.3,virginica
3,5.0,3.4,1.6,0.4,setosa
4,5.7,2.6,3.5,1.0,versicolor


## 2. Schemat walidacyjny

Schemat zakłada pięć kolumn irisowych, wartości numeryczne w sensownych zakresach oraz trzy dopuszczalne gatunki. Zakres `0-30` jest celowo szeroki, żeby wyłapać oczywiste pomyłki skali, a nie poprawiać naturalne różnice biologiczne.

In [27]:
measurement_columns = ["Sepal.Length", "Sepal.Width", "Petal.Length", "Petal.Width"]
species_values = ["setosa", "versicolor", "virginica"]

iris_schema = DataFrameSchema(
    {
        "Sepal.Length": Column(float, Check.between(0, 30), coerce=True),
        "Sepal.Width": Column(float, Check.between(0, 30), coerce=True),
        "Petal.Length": Column(float, Check.between(0, 30), coerce=True),
        "Petal.Width": Column(float, Check.between(0, 30), coerce=True),
        "Species": Column(str, Check.isin(species_values), coerce=True),
    },
    strict=True,
    ordered=True,
)

iris_schema

<Schema DataFrameSchema(columns={'Sepal.Length': <Schema Column(name=Sepal.Length, type=DataType(float64))>, 'Sepal.Width': <Schema Column(name=Sepal.Width, type=DataType(float64))>, 'Petal.Length': <Schema Column(name=Petal.Length, type=DataType(float64))>, 'Petal.Width': <Schema Column(name=Petal.Width, type=DataType(float64))>, 'Species': <Schema Column(name=Species, type=DataType(str))>}, checks=[], parsers=[], index=None, dtype=None, coerce=False, strict=True, name=None, ordered=True, unique=None, report_duplicates=all, unique_column_names=False, add_missing_columns=False, title=None, description=None, metadata=None, drop_invalid_rows=False)>

## 3. Walidacja brudnych danych

Używamy trybu `lazy=True`, żeby dostać pełną listę naruszeń zamiast kończyć na pierwszym błędzie.

In [28]:
try:
    iris_schema.validate(dirty_iris, lazy=True)
    print("Dane przechodzą walidację bez błędów.")
except pa.errors.SchemaErrors as err:
    failure_cases = err.failure_cases
    print(f"Liczba naruszeń: {len(failure_cases)}")
    failure_cases.head(20)

Liczba naruszeń: 63


## 4. Czyszczenie

W tym zbiorze najważniejsze problemy to pomylona skala w części wierszy, brakujące wartości liczbowe, pojedyncze wartości ujemne w kolumnach pomiarowych oraz wartości nieskończone. Jeśli którejkolwiek z czterech miar biologicznych wartość przekracza 30, traktujemy to jako błąd skali i dzielimy przez 10. Braki oraz nieskończoności uzupełniamy medianą kolumny, wartości ujemne zamieniamy na dodatnie przez `abs()`, a zapis gatunku normalizujemy.


In [29]:
def clean_dirty_iris(frame: pd.DataFrame) -> pd.DataFrame:
    cleaned = frame.copy()
    cleaned[measurement_columns] = cleaned[measurement_columns].apply(pd.to_numeric, errors="coerce")
    cleaned[measurement_columns] = cleaned[measurement_columns].replace([np.inf, -np.inf], np.nan)
    cleaned[measurement_columns] = cleaned[measurement_columns].abs()
    cleaned[measurement_columns] = cleaned[measurement_columns].fillna(cleaned[measurement_columns].median())

    scale_mask = cleaned[measurement_columns].gt(30).any(axis=1)
    cleaned.loc[scale_mask, measurement_columns] = cleaned.loc[scale_mask, measurement_columns] / 10

    cleaned["Species"] = cleaned["Species"].astype(str).str.strip().str.lower()
    return cleaned


cleaned_iris = clean_dirty_iris(dirty_iris)
cleaned_iris.head(5)


,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
0,6.4,3.2,4.5,1.5,versicolor
1,6.3,3.3,6.0,2.5,virginica
2,6.2,3.0,5.4,2.3,virginica
3,5.0,3.4,1.6,0.4,setosa
4,5.7,2.6,3.5,1.0,versicolor


## 5. Ponowna walidacja

Po czyszczeniu schemat powinien już przechodzić bez naruszeń.

In [30]:
validated_iris = iris_schema.validate(cleaned_iris)
print("Walidacja po czyszczeniu zakończona sukcesem.")
validated_iris.describe(include="all")

Walidacja po czyszczeniu zakończona sukcesem.


,Sepal.Length,Sepal.Width,Petal.Length,Petal.Width,Species
count,150.000000,150.000000,150.000000,150.000000,150
unique,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,versicolor
freq,NaN,NaN,NaN,NaN,50
mean,5.773333,3.032667,3.994300,1.195533,NaN
std,0.936753,0.448936,2.301653,0.738045,NaN
min,0.000000,0.000000,0.000000,0.100000,NaN
25%,5.100000,2.800000,1.700000,0.300000,NaN
50%,5.750000,3.000000,4.500000,1.300000,NaN
75%,6.375000,3.200000,5.100000,1.800000,NaN


## 6. Krótki efekt końcowy

Na końcu można porównać liczbę wierszy przed i po czyszczeniu oraz przejrzeć gotową ramkę danych.

In [33]:
summary = pd.DataFrame(
    {
        "rows_before": [len(dirty_iris)],
        "rows_after": [len(cleaned_iris)],
        "measurement_columns": [", ".join(measurement_columns)],
    }
)
summary


,rows_before,rows_after,measurement_columns
0,150,150,"Sepal.Length, Sepal.Width, Petal.Length, Petal..."
